# ARC-AGI-3 Kaggle Submission Notebook — Version 2

Notebook orientado a ejecución de agente en múltiples juegos, con score agregado y una política mejorada.

No exporta `submission.json`: el objetivo es que Kaggle ejecute el agente y evalúe su comportamiento.

## 1. Instalación de dependencias

In [ ]:
!pip install -q "arc-agi" "pillow<12"

## 2. Imports

In [ ]:
import json
import random
from collections import defaultdict

import arc_agi


## 3. Configuración
Podés ajustar la lista de juegos y la cantidad máxima de pasos.

In [ ]:
MAX_STEPS = 120
GAME_IDS = ["ls20", "ls21", "ls22", "ls23"]
RANDOM_SEED = 42
random.seed(RANDOM_SEED)

## 4. Agente versión 2
Mejoras sobre la versión inicial:
- penaliza acciones repetidas
- aprende una señal simple de utilidad por reward
- favorece novedad en estados
- usa exploración espacial menos ciega para acciones complejas

In [ ]:
class SubmissionAgentV2:
    def __init__(self, max_steps=120):
        self.max_steps = max_steps
        self.coord_candidates = [
            (0, 0), (0, 15), (15, 0), (15, 15),
            (3, 3), (7, 7), (8, 4), (4, 8), (10, 10),
            (1, 8), (8, 1), (12, 6), (6, 12)
        ]
        self.reset()

    def reset(self):
        self.step_count = 0
        self.visited_states = defaultdict(int)
        self.action_counts = defaultdict(int)
        self.action_success = defaultdict(float)
        self.last_reward = 0.0
        self.coord_index = 0

    def state_key(self, obs):
        return repr(obs)[:1000]

    def novelty_bonus(self, obs):
        if obs is None:
            return 0.0
        key = self.state_key(obs)
        return 1.0 / (1.0 + self.visited_states[key])

    def choose_coord(self):
        coord = self.coord_candidates[self.coord_index % len(self.coord_candidates)]
        self.coord_index += 1
        return coord

    def act(self, env, obs):
        if obs is not None:
            key = self.state_key(obs)
            self.visited_states[key] += 1

        actions = list(env.action_space)
        novelty = self.novelty_bonus(obs)

        def action_score(action):
            name = str(action)
            success_bonus = self.action_success[name]
            usage_penalty = 0.30 * self.action_counts[name]
            exploration_bonus = 0.15 * novelty
            return success_bonus - usage_penalty + exploration_bonus

        actions.sort(key=action_score, reverse=True)

        pool_size = max(1, len(actions) // 2)
        pool = actions[:pool_size]
        action = random.choice(pool)

        action_data = {}
        if hasattr(action, "is_complex") and action.is_complex():
            x, y = self.choose_coord()
            action_data = {"x": x, "y": y}

        self.action_counts[str(action)] += 1
        return action, action_data

    def update(self, action, reward):
        delta = reward - self.last_reward
        self.action_success[str(action)] += delta
        self.last_reward = reward


## 5. Normalización de `env.step`
Compatibilidad con distintas firmas de retorno.

In [ ]:
def parse_step_result(step_result):
    if not isinstance(step_result, tuple):
        return step_result, 0.0, False, False, {}

    if len(step_result) >= 5:
        obs = step_result[0]
        reward = step_result[1]
        terminated = bool(step_result[2])
        truncated = bool(step_result[3])
        info = step_result[4] if isinstance(step_result[4], dict) else {}
        return obs, reward, terminated, truncated, info

    if len(step_result) == 4:
        obs, reward, done, info = step_result
        return obs, reward, bool(done), False, info if isinstance(info, dict) else {}

    if len(step_result) == 3:
        obs, reward, done = step_result
        return obs, reward, bool(done), False, {}

    if len(step_result) == 2:
        obs, reward = step_result
        return obs, reward, False, False, {}

    return step_result[0], 0.0, False, False, {}


## 6. Runner por juego

In [ ]:
def run_single_game(game_id, max_steps=120):
    arc = arc_agi.Arcade()
    env = arc.make(game_id, render_mode=None)
    agent = SubmissionAgentV2(max_steps=max_steps)

    obs = None
    reward = 0.0
    terminated = False
    truncated = False
    info = {}
    trace = []

    for step in range(agent.max_steps):
        action, action_data = agent.act(env, obs)
        step_result = env.step(action, data=action_data)
        obs, reward, terminated, truncated, info = parse_step_result(step_result)
        agent.update(action, reward)

        trace.append({
            "step": step,
            "action": str(action),
            "action_data": action_data,
            "reward": reward,
            "terminated": terminated,
            "truncated": truncated,
            "info": info
        })

        if terminated or truncated:
            break

    scorecard = arc.get_scorecard()
    return {
        "game_id": game_id,
        "steps": len(trace),
        "final_reward": reward,
        "scorecard": str(scorecard),
        "trace": trace
    }


## 7. Runner batch
Ejecuta múltiples juegos y agrega métricas simples.

In [ ]:
def run_batch(game_ids, max_steps=120):
    results = []
    for game_id in game_ids:
        try:
            result = run_single_game(game_id, max_steps=max_steps)
            results.append(result)
            print(f"OK  {game_id} | steps={result['steps']} | final_reward={result['final_reward']}")
        except Exception as e:
            results.append({
                "game_id": game_id,
                "error": str(e)
            })
            print(f"ERR {game_id} | {e}")
    return results


## 8. Ejecución sobre varios juegos

In [ ]:
batch_results = run_batch(GAME_IDS, max_steps=MAX_STEPS)
batch_results

## 9. Resumen agregado
Resumen simple para inspección rápida en Kaggle.

In [ ]:
successful = [r for r in batch_results if 'error' not in r]
failed = [r for r in batch_results if 'error' in r]

summary = {
    "games_requested": len(GAME_IDS),
    "games_succeeded": len(successful),
    "games_failed": len(failed),
    "avg_steps": sum(r['steps'] for r in successful) / len(successful) if successful else None,
    "avg_final_reward": sum(r['final_reward'] for r in successful) / len(successful) if successful else None
}

summary

## 10. Nota final
Esta versión 2 elimina el export artificial a `submission.json` y se enfoca en ejecución, trazabilidad y score agregado.